### 1. Setup and configuration
All knobs are in `CFG`. `P7_CFG_OVERRIDE` (JSON in an env var) overrides them without editing the cell.

In [ ]:

# CELL 1 - setup, configuration, small utilities

import os, sys, json, re, math, time, csv, random, zipfile, shutil, hashlib, warnings, gc, types, subprocess
from dataclasses import dataclass, asdict, fields
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np

def _ensure(pip_name, import_name):
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

_ensure("tokenizers", "tokenizers")
_ensure("huggingface_hub", "huggingface_hub")
_ensure("scikit-learn", "sklearn")
_ensure("pandas", "pandas")

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

warnings.filterwarnings("ignore", category=UserWarning)

# CONFIG. Everything you may want to change is here.
# Quick smoke run on Kaggle without editing the cell:
# os.environ["P7_CFG_OVERRIDE"] = '{"EPOCHS":1,"MAX_QUERIES":40,"MAX_POOL":400}'

CFG = dict(
    EXP_NAME="m1_lormaplus_multiview_fusion",
    P7_TAG="p7_m1_lormaplus_multiview_fusion_v1",          # tag stored in this notebook's checkpoints
    P1_TAG=None,                # (Model-1 checkpoints carry no tag)

    # where things are (None = auto-discover under /kaggle/input)
    P1_CKPT=None,               # path to the Phase-1 Model-1 best.pt
    P1_VOCAB_DIR=None,          # folder with the Model-1 BPE vocab.json + merges.json
    P7_ROOT=None,               # folder that contains Phase_7_PCR/ (None = download from Hugging Face)
    HF_DATASET_REPO="Exploration-Lab/CS-781-Capstone",

    # LONG-CONTEXT HANDLING (varies between the experiment files)
    STRIP_PLACEHOLDERS=True,    # remove <NAME>, <ORG>, <CITATION> ... anonymisation tags from the text
    MAX_CHARS=100_000,          # characters kept per case (head of the document)
    CHUNK_LEN=512,         # window length in tokens; must be <= the Phase-1 max_len (1088)
    CHUNK_STRIDE=384,      # == CHUNK_LEN -> no overlap between windows
    MIN_TAIL=64,                # a trailing window shorter than this is dropped
    MAX_CHUNKS=48,          # windows per case at inference
    SECTION_HEAD=0,           # >0 or SECTION_TAIL>0: keep only the first SECTION_HEAD and the last SECTION_TAIL windows of a case
    SECTION_TAIL=0,
    TOKEN_POOL="mean",          # inside a window:  mean | last   ("last" is far worse for this causal LM)
    DOC_POOL="mean",            # across windows:   mean | max | first | attn | maxsim  (maxsim = late interaction, scores of window sets)
    CENTER="none",              # none | pool  (subtract the evaluated pool's mean embedding; not used with maxsim)
    PROJ_DIM=None,              # None = d_model (identity-initialised projection head)
    TFIDF_NGRAM=(1, 1),         # (1, 2) uses a lot more RAM

    PEFT="lorma_plus",           # lorma_plus | lorma_pi | partial | ia3 | full
    LORA_R=32, LORA_ALPHA=32,       # LoRMA rank / scale (scale = alpha / r, used by LoRMA+ only)
    LORA_TARGETS=("q_proj", "k_proj", "v_proj", "o_proj", "fc1", "fc2"),
    TRAIN_TOP_BLOCKS=3,       # partial FT: how many blocks (from the top) are trained
    LAYER_DECAY=1.0,         # LR of block i = LR * LAYER_DECAY ** (distance from the top block)
    BB_WD=0.0,                # weight decay on backbone matrices
    NORM_LR_MULT=1.0,         # LR multiplier of the RMSNorm gains (only matters if they are trainable: IA3 / partial / full)

    # RANKING OBJECTIVE (varies between the experiment files)
    LOSS="combo",           # infonce_mp | margin | listwise | lambdarank | combo (= infonce_mp + COMBO_W * margin)
    N_POS=2,                 # gold cases sampled per query and step
    MARGIN=0.1, COMBO_W=1.0,
    TEMP=0.05,                  # softmax / logistic temperature

    # training
    TRAIN=True,
    EPOCHS=8,
    BATCH_Q=8,                  # queries per step (each brings N_POS positives + HARD_NEGS hard negatives)
    HARD_NEGS=3,
    HARD_POOL=30,               # hard negatives are sampled from the TF-IDF top-HARD_POOL (gold removed)
    TRAIN_CHUNKS=3,             # random windows per document at train time
    LR=0.0003, HEAD_LR=5e-05, WEIGHT_DECAY=0.01, WARMUP_FRAC=0.1, MIN_LR_RATIO=0.1, GRAD_CLIP=1.0,
    GRAD_CKPT=True,
    TOK_BUDGET_TRAIN=16384,     # tokens per forward micro-batch while training (lower if CUDA OOM)
    TOK_BUDGET_EVAL=32768,      # tokens per forward micro-batch at inference
    EVAL_EVERY=1,               # dev evaluation every N epochs
    PATIENCE=3,                 # stop after N dev evaluations without improvement
    MAX_RUNTIME_H=10.5,

    # multi-view fusion (only used when FUSE=True)
    FUSE=True,
    FUSE_MIN_GAIN=0.003,        # a blend / feedback step must beat the previous dev F1@7 by at least this much
    EXTRA_SCORES=[],            # optional: paths of scores_*.npz written by OTHER experiment files (same split order) -> ensembling

    # evaluation / output
    K=7,
    SEED=1234,
    RUN_ZERO_SHOT=True,         # analysis: Phase-1 model without any adaptation
    RUN_PREDICT=True,
    USE_HYBRID=False,           # True = final ranking = dense + lambda * TF-IDF (lambda picked on dev)
    MAX_QUERIES=None,           # smoke-test switches (per split)
    MAX_POOL=None,
)
_ov = os.environ.get("P7_CFG_OVERRIDE")
if _ov:
    CFG.update(json.loads(_ov))
CFG["LORA_TARGETS"] = tuple(CFG["LORA_TARGETS"])
CFG["EXTRA_SCORES"] = list(CFG["EXTRA_SCORES"])
CFG["TFIDF_NGRAM"] = tuple(CFG["TFIDF_NGRAM"])
C = types.SimpleNamespace(**CFG)
assert C.TOKEN_POOL in ("mean", "last") and C.DOC_POOL in ("mean", "max", "first", "attn", "maxsim") and C.CENTER in ("none", "pool")
assert C.DOC_POOL != "maxsim" or C.CENTER == "none", "centring is only defined for single-vector documents"
assert C.PEFT in ("lorma_plus", "lorma_pi", "partial", "ia3", "full") and C.LOSS in ("infonce_mp", "margin", "listwise", "lambdarank", "combo")
assert C.CHUNK_LEN <= 1088 and C.CHUNK_STRIDE >= 1 and C.N_POS >= 1

# paths / device

IS_KAGGLE = os.path.isdir("/kaggle/working")
INPUT_ROOT = "/kaggle/input"
BASE_DIR = "/kaggle/working" if IS_KAGGLE else "."
WORK_DIR = os.path.join(BASE_DIR, "p7_" + C.EXP_NAME)
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints")
OUT_DIR = os.path.join(BASE_DIR, "cs781_phase7_" + C.EXP_NAME)       # prediction.json + submission.zip land here
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = {"enabled": False, "dtype": torch.float16}
if DEVICE.type == "cuda":
    AMP["enabled"] = True
    AMP["dtype"] = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=AMP["dtype"], enabled=AMP["enabled"])

def log(*a):
    print(*a, flush=True)

def fmt_time(sec):
    sec = int(max(sec, 0))
    return f"{sec // 3600}h{(sec % 3600) // 60:02d}m{sec % 60:02d}s"

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def walk_find(filename, roots):
    """all directories under `roots` that contain a file called `filename`"""
    hits, seen = [], set()
    for root in roots:
        if not root or not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            if filename in files and d not in seen:
                seen.add(d)
                hits.append(d)
    return hits

def read_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)

def atomic_torch_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def hf_token():
    # Kaggle secret named HF, or env HF_TOKEN / HF. Nothing is hard-coded in this notebook.
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF")
        if t:
            return t
    except Exception:
        pass
    return os.environ.get("HF_TOKEN") or os.environ.get("HF") or None

HF_TOKEN = hf_token()
set_seed(C.SEED)
RESULTS = {"config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()}}
T_START = time.time()

log(f"torch {torch.__version__} | device {DEVICE} | "
    f"{torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'NO GPU (this will be very slow)'} | "
    f"autocast {AMP['dtype'] if AMP['enabled'] else 'off'}")
log(f"HF token available: {bool(HF_TOKEN)} | work dir {WORK_DIR} | output dir {OUT_DIR}")

### 2. Phase-1 Model 1 (encoder) and tokenizer
The architecture classes are copied from the Phase-1 Model-1 notebook so the checkpoint loads without renaming (generation helpers left out). Only the **encoder** is used for retrieval; the decoder is dropped right after loading.

In [ ]:


WORD_PATTERN = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d|[a-zA-Z]+|[0-9]+|[^\s\a-zA-Z0-9]+""")
_ENC_TXT = []

def _enc_one(i):
    return i, np.asarray(TOK.encode(_ENC_TXT[i]), dtype=np.int32)

class LegalBPE:
    """Phase-1 Model-1 tokenizer (vocab.json + merges.json): lower-casing word-level BPE with </w> end markers, pure python"""

    def __init__(self, vocab_path, merges_path, meta_path=None):
        with open(vocab_path, "r", encoding="utf-8") as f:
            self.vocab = json.load(f)
        with open(merges_path, "r", encoding="utf-8") as f:
            merges = json.load(f)
        self.ranks = {(a, b): i for i, (a, b) in enumerate(merges)}
        self.inv = {int(i): t for t, i in self.vocab.items()}
        self.vocab_size = max(self.inv) + 1
        assert len(self.inv) == self.vocab_size, "vocab ids are not contiguous"
        meta = {}
        if meta_path and os.path.exists(meta_path):
            with open(meta_path, "r") as f:
                meta = json.load(f)
        self.pad_id = self.vocab["<|pad|>"]
        self.unk_id = self.vocab["<|unk|>"]
        self.eos_id = self.vocab["<|endoftext|>"]
        for key, val in (("pad_id", self.pad_id), ("unk_id", self.unk_id), ("eos_id", self.eos_id)):
            if key in meta and meta[key] != val:
                raise ValueError(f"meta.json {key}={meta[key]} disagrees with vocab.json ({val})")
        if meta.get("vocab_size") not in (None, self.vocab_size):
            log(f"warning: meta.json vocab_size={meta['vocab_size']} but vocab.json has {self.vocab_size}")
        self.bos_id = self.eos_id                      # no BOS in this vocab: the decoder start symbol reuses <|endoftext|>
        self.cache = {}

    def _word_ids(self, word):
        hit = self.cache.get(word)
        if hit is not None:
            return hit
        syms = list(word) + ["</w>"]
        while len(syms) > 1:
            best, best_rank = None, None
            for i in range(len(syms) - 1):
                r = self.ranks.get((syms[i], syms[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = (syms[i], syms[i + 1]), r
            if best is None:
                break
            merged = best[0] + best[1]
            out, i = [], 0
            while i < len(syms):
                if i < len(syms) - 1 and syms[i] == best[0] and syms[i + 1] == best[1]:
                    out.append(merged)
                    i += 2
                else:
                    out.append(syms[i])
                    i += 1
            syms = out
        ids = [self.vocab.get(s, self.unk_id) for s in syms]
        if len(self.cache) > 3_000_000:
            self.cache.clear()
        self.cache[word] = ids
        return ids

    def encode(self, text):
        ids = []
        for w in WORD_PATTERN.findall(text.lower()):
            ids.extend(self._word_ids(w))
        return ids

    def encode_batch(self, texts):
        """list of texts -> list of int32 arrays.  The pure-python BPE is slow, so the documents are encoded in forked worker
        processes (each keeps its own word cache for the life of the pool); falls back to one process if forking fails."""
        global _ENC_TXT
        _ENC_TXT = texts
        out = [None] * len(texts)
        n_proc = max(1, min(os.cpu_count() or 1, 4))
        if n_proc > 1 and len(texts) > 64:
            try:
                import multiprocessing as mp
                order = sorted(range(len(texts)), key=lambda i: -len(texts[i]))        # longest first -> balanced workers
                with mp.get_context("fork").Pool(n_proc) as pool:
                    for n, (i, a) in enumerate(pool.imap_unordered(_enc_one, order, chunksize=8), 1):
                        out[i] = a
                        if n % 2000 == 0:
                            log(f"    tokenised {n}/{len(texts)} documents")
            except Exception as e:
                log(f"  parallel tokenisation failed ({e!r}); continuing in one process")
        for i in range(len(texts)):
            if out[i] is None:
                out[i] = _enc_one(i)[1]
        return out

@dataclass
class ModelConfig:
    vocab_size: int = 35000
    d_model: int = 768
    n_layers: int = 6
    n_heads: int = 12
    n_kv_heads: int = 4
    ffn_dim: int = 3072
    dropout: float = 0.2
    max_len: int = 1024
    pad_id: int = 0
    bos_id: int = 2
    eos_id: int = 2
    unk_id: int = 1

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).type_as(x)

def build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()          # each (max_len, head_dim/2)

def apply_rope(x, cos, sin):
    # x: (B, H, T, hd), cos/sin: (T, hd/2)
    x1, x2 = x.float().chunk(2, dim=-1)
    cos, sin = cos[None, None], sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)

def repeat_kv(x, groups):
    if groups == 1:
        return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)

class GQASelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.n_heads
        self.kv = cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)

    def _qkv(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        return q, k, v

    def forward(self, x, cos, sin, causal, pad_mask=None):
        B, T, _ = x.shape
        q, k, v = self._qkv(x)
        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])
        k, v = repeat_kv(k, self.groups), repeat_kv(v, self.groups)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=pad_mask, is_causal=(causal and pad_mask is None))
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))

class CrossAttention(nn.Module):
    """plain multi-head attention, queries from the decoder, keys/values from the encoder (only needed to load the checkpoint)"""

    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.n_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.o_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)

class FFN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.ffn_dim)
        self.fc2 = nn.Linear(cfg.ffn_dim, cfg.d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = GQASelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn = FFN(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin, mask):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin, False, mask))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x

class DecoderLayer(nn.Module):
    """parameter layout only (so that the Phase-1 state_dict loads with strict matching); never executed here"""

    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.self_attn = GQASelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.cross_attn = CrossAttention(cfg)
        self.norm3 = RMSNorm(cfg.d_model)
        self.ffn = FFN(cfg)
        self.drop = nn.Dropout(cfg.dropout)

class EncDecLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)     # shared by encoder, decoder and output head
        self.enc_layers = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.n_layers)])
        self.dec_layers = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.n_layers)])
        self.enc_norm = RMSNorm(cfg.d_model)
        self.dec_norm = RMSNorm(cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    @staticmethod
    def _mask4(src_mask):

        if src_mask is None or bool(src_mask.all()):
            return None
        return src_mask[:, None, None, :]

def count_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if (p.requires_grad or not trainable_only))

# find + load the Phase-1 Model-1 checkpoint and tokenizer

def find_phase1_ckpt():
    """Model-1 output of the Phase-1 notebook: best.pt (+ best_meta.json WITHOUT a 'tag' key); falls back to latest.pt"""
    if C.P1_CKPT:
        assert os.path.isfile(C.P1_CKPT), f"P1_CKPT not found: {C.P1_CKPT}"
        return C.P1_CKPT
    roots = [INPUT_ROOT, "/kaggle/working", os.getcwd()]
    best = None
    for d in walk_find("best_meta.json", roots):
        meta = read_json(os.path.join(d, "best_meta.json"))
        if not meta or "tag" in meta or "val_loss" not in meta or not os.path.exists(os.path.join(d, "best.pt")):
            continue                                   # a 'tag' key = a decoder-only Model-2 checkpoint
        if best is None or meta["val_loss"] < best[1]["val_loss"]:
            best = (d, meta)
    if best is not None:
        log(f"Phase-1 Model-1 best.pt: {best[0]} (val loss {best[1]['val_loss']:.4f})")
        return os.path.join(best[0], "best.pt")
    latest = None
    for d in walk_find("latest_meta.json", roots):
        meta = read_json(os.path.join(d, "latest_meta.json"))
        if not meta or "tag" in meta or not os.path.exists(os.path.join(d, "latest.pt")):
            continue
        if latest is None or meta["global_step"] > latest[1]["global_step"]:
            latest = (d, meta)
    if latest is not None:
        log(f"Phase-1 Model-1 latest.pt: {latest[0]}")
        return os.path.join(latest[0], "latest.pt")
    raise FileNotFoundError("Phase-1 Model-1 checkpoint (best.pt / latest.pt) not found - attach the Model-1 output as a Kaggle input, or set P1_CKPT")

def load_tokenizer():
    """Phase-1 BPE (vocab.json + merges.json [+ meta.json]); searches /kaggle/input unless P1_VOCAB_DIR is set"""
    roots = [C.P1_VOCAB_DIR] if C.P1_VOCAB_DIR else [INPUT_ROOT, os.getcwd(), "/kaggle/working"]
    for d in walk_find("vocab.json", roots):
        if not os.path.exists(os.path.join(d, "merges.json")):
            continue
        try:
            tok = LegalBPE(os.path.join(d, "vocab.json"), os.path.join(d, "merges.json"), os.path.join(d, "meta.json"))
        except Exception as e:                         # some other model's vocab.json (no <|pad|> ...)
            log(f"skipping {d}: {e!r}")
            continue
        log(f"tokenizer: {d} | vocab={tok.vocab_size} pad={tok.pad_id} unk={tok.unk_id} eos={tok.eos_id}")
        return tok
    raise FileNotFoundError("Phase-1 BPE vocab.json + merges.json not found - attach the BPE dataset as a Kaggle input, or set P1_VOCAB_DIR")

def load_phase1(tok):
    ck_path = find_phase1_ckpt()
    log(f"Phase-1 checkpoint: {ck_path}")
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    names = {f.name for f in fields(ModelConfig)}
    raw = {k: v for k, v in (ck.get("config") or {}).items() if k in names}
    if not raw:
        raw = dict(vocab_size=tok.vocab_size, pad_id=tok.pad_id, bos_id=tok.bos_id, eos_id=tok.eos_id, unk_id=tok.unk_id, max_len=896)
    cfg = ModelConfig(**raw)
    cfg.dropout = 0.0                                              # no dropout in the frozen base
    assert cfg.vocab_size == tok.vocab_size, f"tokenizer vocab {tok.vocab_size} != checkpoint vocab {cfg.vocab_size}: wrong BPE attached?"
    sd = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in ck["model"].items()}
    if not any(k.startswith("enc_layers.") for k in sd):
        raise RuntimeError("this checkpoint has no encoder (enc_layers.*) - it is not the Phase-1 Model-1 encoder-decoder")
    model = EncDecLM(cfg)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    missing = [m for m in missing if not m.startswith("rope_")]
    assert not missing and not unexpected, f"state-dict mismatch: missing={missing} unexpected={unexpected}"
    n_full = count_params(model)
    model.dec_layers = nn.ModuleList()                             # the decoder is not needed for retrieval
    model.dec_norm = nn.Identity()
    log(f"loaded Phase-1 Model 1 strictly | {n_full / 1e6:.1f}M params in total, {count_params(model) / 1e6:.1f}M used "
        f"(encoder + tied embeddings) | d_model={cfg.d_model} enc layers={cfg.n_layers} heads={cfg.n_heads}/{cfg.n_kv_heads} max_len={cfg.max_len}")
    return model, cfg, ck_path, ck.get("tag")

TOK = load_tokenizer()
P1_MODEL, P1_CFG, P1_CKPT_PATH, P1_CKPT_TAG = load_phase1(TOK)
assert C.CHUNK_LEN <= P1_CFG.max_len, f"CHUNK_LEN {C.CHUNK_LEN} exceeds the Phase-1 max_len {P1_CFG.max_len}"
if C.CHUNK_LEN > 768:
    log("note: the Model-1 encoder was pre-trained on <= 768-token inputs; longer windows are out of distribution")
RESULTS["phase1"] = {"ckpt": P1_CKPT_PATH, "tag": P1_CKPT_TAG, "params": count_params(P1_MODEL)}

### 3. Phase-7 data
Queries and candidates per split. Sentence lists are joined in order; anonymisation tags (`<NAME>`, `<ORG>`, `<CITATION>`) are stripped by default.

In [ ]:

# CELL 3 - Phase-7 data: load, normalise, per-split structures

P7_FILES = {
    "train_queries": "Phase_7_PCR/Train/train.json",
    "train_candidates": "Phase_7_PCR/Train/train_candidates.json",
    "dev_queries": "Phase_7_PCR/Dev/dev.json",
    "dev_candidates": "Phase_7_PCR/Dev/dev_candidates.json",
    "test_queries": "Phase_7_PCR/Test/test.json",
    "test_candidates": "Phase_7_PCR/Test/test_candidates.json",
}

def _looks_like_p7(path):
    p = path.lower().replace("-", "_")
    return "phase_7" in p or "phase7" in p or "pcr" in p

def p7_file(rel):
    """local copy (P7_ROOT / Kaggle input) first, then the Hugging Face dataset repo"""
    base = os.path.basename(rel)
    if C.P7_ROOT:
        for cand in (Path(C.P7_ROOT) / rel, Path(C.P7_ROOT) / base):
            if cand.is_file():
                return str(cand)
        for d in walk_find(base, [C.P7_ROOT]):
            return os.path.join(d, base)
    for d in walk_find(base, [INPUT_ROOT]):                         # an attached copy of the dataset
        if _looks_like_p7(d):
            return os.path.join(d, base)
    err = None
    try:
        from huggingface_hub import hf_hub_download
        return hf_hub_download(repo_id=C.HF_DATASET_REPO, filename=rel, repo_type="dataset", token=HF_TOKEN)
    except Exception as e:
        err = e
    raise FileNotFoundError(f"cannot find {rel}: not under P7_ROOT / {INPUT_ROOT} and the HF download failed: {err!r}")

def unwrap_json(obj):
    if isinstance(obj, dict):
        for key in ("data", "records", "examples", "queries", "candidates"):
            if isinstance(obj.get(key), (list, dict)):
                return obj[key]
    return obj

def normalize_text(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):                                     # sentence list -> one string, order preserved
        return " ".join(str(x).strip() for x in value if x is not None).strip()
    return str(value).strip()

def records_from_queries(obj):
    """-> list of (query_id, text, [relevant ids])"""
    rows = []
    if isinstance(obj, list):
        iterator = enumerate(obj)
    elif isinstance(obj, dict):
        iterator = [(None, obj)] if any(k in obj for k in ("id", "case_id", "query_id")) else obj.items()
    else:
        raise TypeError(f"queries must be list/dict, got {type(obj)}")
    for key, rec in iterator:
        if not isinstance(rec, dict):
            continue
        cid = rec.get("id", rec.get("case_id", rec.get("query_id", key)))
        text = rec.get("text", rec.get("document", rec.get("case_text", "")))
        rel = rec.get("relevant_candidates", rec.get("relevant_candidate_ids", [])) or []
        if isinstance(rel, str):
            rel = [rel]
        rows.append((str(cid), normalize_text(text), [str(x) for x in rel]))
    return rows

def records_from_candidates(obj):
    """-> dict candidate_id -> text"""
    out = {}
    if isinstance(obj, dict):
        iterable = [(None, obj)] if any(k in obj for k in ("id", "case_id", "text", "document")) else obj.items()
    elif isinstance(obj, list):
        iterable = [(None, r) for r in obj]
    else:
        raise TypeError(f"candidates must be list/dict, got {type(obj)}")
    for key, rec in iterable:
        if isinstance(rec, dict):
            cid = rec.get("id", rec.get("case_id", rec.get("candidate_id", key)))
            txt = rec.get("text", rec.get("document", rec.get("case_text", "")))
        else:
            cid, txt = key, rec
        if cid is None:
            raise ValueError("cannot infer a candidate id - inspect the raw candidate file")
        out[str(cid)] = normalize_text(txt)
    return out

_PLACEHOLDER = re.compile(r"<\s*[A-Z][A-Z_]{1,19}\s*>")             # <NAME> <ORG> <CITATION> ...

def prep_text(t):
    if C.STRIP_PLACEHOLDERS:
        t = _PLACEHOLDER.sub(" ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t[:C.MAX_CHARS]

def limit_split(sp, max_q, max_pool, seed):
    """smoke-test helper: first `max_q` queries; pool = their gold + their own ids + random fill up to `max_pool`"""
    rng = random.Random(seed)
    qids = sp.qids[:max_q] if max_q else list(sp.qids)
    pool = set(sp.cids)
    keep = set()
    for q in qids:
        keep |= (sp.gold[q] & pool)
        if q in pool:
            keep.add(q)
    rest = [c for c in sp.cids if c not in keep]
    rng.shuffle(rest)
    n_fill = len(rest) if not max_pool else max(max_pool - len(keep), 0)
    keep |= set(rest[:n_fill])
    sp.qids = qids
    sp.cids = [c for c in sp.cids if c in keep]
    sp.qtext = {q: sp.qtext[q] for q in qids}
    sp.gold = {q: sp.gold[q] for q in qids}
    sp.ctext = {c: sp.ctext[c] for c in sp.cids}
    return sp

def load_split(name):
    q_raw = unwrap_json(json.load(open(p7_file(P7_FILES[f"{name}_queries"]), encoding="utf-8")))
    c_raw = unwrap_json(json.load(open(p7_file(P7_FILES[f"{name}_candidates"]), encoding="utf-8")))
    qrows = records_from_queries(q_raw)
    cmap = records_from_candidates(c_raw)
    sp = types.SimpleNamespace(name=name)
    sp.qids = [q for q, _, _ in qrows]
    sp.qtext = {q: prep_text(t) for q, t, _ in qrows}
    sp.gold = {q: set(r) for q, _, r in qrows}
    sp.cids = list(cmap.keys())
    sp.ctext = {c: prep_text(t) for c, t in cmap.items()}
    if C.MAX_QUERIES or C.MAX_POOL:
        sp = limit_split(sp, C.MAX_QUERIES, C.MAX_POOL, C.SEED)
        log(f"[smoke] {name}: limited to {len(sp.qids)} queries / {len(sp.cids)} candidates")
    pool = set(sp.cids)
    sp.gold_in_pool = {q: sorted((sp.gold[q] & pool) - {q}) for q in sp.qids}
    return sp

SP = {name: load_split(name) for name in ("train", "dev", "test")}

rows = []
for name, sp in SP.items():
    pool = set(sp.cids)
    ng = [len(sp.gold[q]) for q in sp.qids]
    nl = [len(sp.qtext[q].split()) for q in sp.qids]
    cl = [len(t.split()) for t in sp.ctext.values()]
    mism = sum(1 for q in sp.qids if q in sp.ctext and sp.ctext[q] != sp.qtext[q])
    rows.append({
        "split": name, "queries": len(sp.qids), "pool": len(sp.cids),
        "query_in_pool": sum(q in pool for q in sp.qids), "text_differs": mism,
        "gold_mean": round(float(np.mean(ng)), 2) if ng else 0, "gold_min": min(ng) if ng else 0,
        "gold_max": max(ng) if ng else 0,
        "gold_not_in_pool": sum(len(sp.gold[q] - pool) for q in sp.qids),
        "no_gold_queries": sum(len(sp.gold[q]) == 0 for q in sp.qids),
        "q_words_med": int(np.median(nl)) if nl else 0, "c_words_med": int(np.median(cl)) if cl else 0,
        "c_words_p95": int(np.percentile(cl, 95)) if cl else 0,
    })
log(pd.DataFrame(rows).to_string(index=False))
log("(query_in_pool > 0 means a query's own case is also in the pool: it is excluded from its own ranking)")
assert all(len(SP[s].qids) > 0 and len(SP[s].cids) > C.K for s in SP), "a split is empty / its pool is smaller than K+1"
assert sum(len(SP["train"].gold_in_pool[q]) > 0 for q in SP["train"].qids) > 0, "no train query has a gold candidate in the pool"
assert any(len(SP["dev"].gold[q]) > 0 for q in SP["dev"].qids), "dev has no labels"
RESULTS["data"] = rows
sample_txt = next(iter(SP["train"].qtext.values()))
log("sample (first 300 chars of a prepared train query):", sample_txt[:300])

### 4. Tokenisation and chunking of long cases

In [ ]:

# CELL 4 - tokenisation and chunking of long cases
# A case becomes up to MAX_CHUNKS chunks of <= CHUNK_LEN tokens (Phase-1 context length).

def chunk_spans(n, L, stride, max_chunks, min_tail):
    """token spans [(start, end), ...] covering a document of n tokens"""
    if n <= L:
        return [(0, n)]
    spans, s = [], 0
    while s < n and len(spans) < max_chunks:
        e = min(s + L, n)
        if spans and e - s < min_tail:                              # drop a tiny trailing chunk
            break
        spans.append((s, e))
        if e == n:
            break
        s += stride
    return spans

def _selftest_chunk_spans():
    assert chunk_spans(10, 1024, 1024, 16, 64) == [(0, 10)]
    assert chunk_spans(1024, 1024, 1024, 16, 64) == [(0, 1024)]
    assert chunk_spans(2048, 1024, 1024, 16, 64) == [(0, 1024), (1024, 2048)]
    assert chunk_spans(2048 + 30, 1024, 1024, 16, 64) == [(0, 1024), (1024, 2048)]           # tail < 64 dropped
    assert chunk_spans(2048 + 100, 1024, 1024, 16, 64) == [(0, 1024), (1024, 2048), (2048, 2148)]
    assert len(chunk_spans(10**6, 1024, 1024, 16, 64)) == 16
    assert chunk_spans(3000, 1024, 512, 16, 64)[-1][1] == 3000                                # overlap: reaches the end
    return True

_selftest_chunk_spans()

def select_sections(spans, head, tail):
    """long-context policy 'head + tail': keep only the first `head` and the last `tail` windows of a case (0 / 0 = keep all)"""
    if (head <= 0 and tail <= 0) or len(spans) <= head + tail:
        return spans
    return spans[:head] + (spans[-tail:] if tail > 0 else [])

def _selftest_sections():
    sp = [(i, i + 1) for i in range(10)]
    assert select_sections(sp, 0, 0) == sp and select_sections(sp, 2, 2) == sp[:2] + sp[-2:]
    assert select_sections(sp[:4], 2, 2) == sp[:4] and select_sections(sp, 3, 0) == sp[:3]
    return True

_selftest_sections()

MAX_TOKENS = C.CHUNK_LEN + (C.MAX_CHUNKS - 1) * C.CHUNK_STRIDE      # hard cap on tokens kept per document

class Doc:
    __slots__ = ("ids", "spans")

    def __init__(self, ids):
        if len(ids) == 0:
            ids = np.array([TOK.unk_id], dtype=np.int32)             # never feed an empty sequence
        self.ids = ids
        self.spans = select_sections(chunk_spans(len(ids), C.CHUNK_LEN, C.CHUNK_STRIDE, C.MAX_CHUNKS, C.MIN_TAIL),
                                     C.SECTION_HEAD, C.SECTION_TAIL)

_TOK_CACHE = {}

def build_docs(texts, batch=100_000):      # one parallel call per split: the pure-python BPE keeps its word cache for the life of the pool
    keys = [hashlib.md5(t.encode("utf-8")).hexdigest() for t in texts]
    todo = {}
    for k, t in zip(keys, texts):
        if k not in _TOK_CACHE and k not in todo:
            todo[k] = t
    items = list(todo.items())
    for i in range(0, len(items), batch):
        part = items[i:i + batch]
        enc = TOK.encode_batch([t for _, t in part])
        for (k, _), ids in zip(part, enc):
            _TOK_CACHE[k] = Doc(np.asarray(ids[:MAX_TOKENS], dtype=np.int32))
    return [_TOK_CACHE[k] for k in keys]

t0 = time.time()
for name, sp in SP.items():
    sp.qdocs = build_docs([sp.qtext[q] for q in sp.qids])
    sp.cdocs = build_docs([sp.ctext[c] for c in sp.cids])
    sp.q_index = {q: i for i, q in enumerate(sp.qids)}
    sp.c_index = {c: i for i, c in enumerate(sp.cids)}
log(f"tokenised {len(_TOK_CACHE):,} unique documents in {fmt_time(time.time() - t0)}")

rows = []
for name, sp in SP.items():
    for role, docs in (("query", sp.qdocs), ("candidate", sp.cdocs)):
        ntok = np.array([len(d.ids) for d in docs])
        nch = np.array([len(d.spans) for d in docs])
        rows.append({"split": name, "role": role, "docs": len(docs), "tok_median": int(np.median(ntok)),
                     "tok_p95": int(np.percentile(ntok, 95)), "tok_max": int(ntok.max()),
                     "chunks_mean": round(float(nch.mean()), 2), "chunks_max": int(nch.max()),
                     "at_token_cap": int((ntok >= MAX_TOKENS).sum()), "total_chunks": int(nch.sum())})
log(pd.DataFrame(rows).to_string(index=False))
RESULTS["tokenisation"] = rows

### 5. Retrieval utilities (ranking, micro-P/R/F1@7, TF-IDF, hard-negative mining)
Pure numpy / scikit-learn. Includes small self-tests.

In [ ]:


def l2n(x, eps=1e-8):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), eps)

def mask_self(S, qids, cids):
    """-inf where a candidate is the query itself (queries also sit in their pool). Works on a copy."""
    S = S.astype(np.float32, copy=True)
    cidx = {c: i for i, c in enumerate(cids)}
    for i, q in enumerate(qids):
        j = cidx.get(q)
        if j is not None:
            S[i, j] = -np.inf
    return S

def rank_topk(S, qids, cids, k):
    """top-k candidate ids per query (best first), the query's own id excluded.  S: [n_queries, n_candidates]"""
    S = mask_self(S, qids, cids)
    k = min(k, S.shape[1] - 1)
    part = np.argpartition(-S, k - 1, axis=1)[:, :k]
    sc = np.take_along_axis(S, part, axis=1)
    order = np.argsort(-sc, axis=1, kind="stable")
    top = np.take_along_axis(part, order, axis=1)
    return {q: [cids[j] for j in top[i]] for i, q in enumerate(qids)}

def micro_prf(preds, gold, qids, k=7):
    """micro P / R / F1@k against the FULL gold set (no truncation, no padding)"""
    tp = n_pred = n_gold = 0
    for q in qids:
        p = list(preds[q][:k])
        g = gold[q]
        tp += len(set(p) & g)
        n_pred += len(p)
        n_gold += len(g)
    P = tp / n_pred if n_pred else 0.0
    R = tp / n_gold if n_gold else 0.0
    F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.0
    return {"P": P, "R": R, "F1": F1, "tp": tp, "n_pred": n_pred, "n_gold": n_gold}

def _selftest_metrics():
    preds = {"a": ["1", "2", "3", "4", "5", "6", "7"], "b": ["1", "2", "3", "4", "5", "6", "7"]}
    gold = {"a": {"1", "2", "9"}, "b": {"7"}}
    m = micro_prf(preds, gold, ["a", "b"], 7)
    assert (m["tp"], m["n_pred"], m["n_gold"]) == (3, 14, 4)
    assert abs(m["P"] - 3 / 14) < 1e-12 and abs(m["R"] - 3 / 4) < 1e-12
    assert abs(m["F1"] - 2 * (3 / 14) * (3 / 4) / (3 / 14 + 3 / 4)) < 1e-12
    S = np.array([[0.9, 0.1, 0.5, 0.8], [0.2, 0.3, 0.1, 0.95]], dtype=np.float32)
    top = rank_topk(S, ["x", "y"], ["x", "b", "c", "d"], 2)          # query "x" is in the pool at column 0
    assert top["x"] == ["d", "c"] and top["y"] == ["d", "b"], top
    return True

_selftest_metrics()

def first_hit_rank(S, qids, cids, gold):
    """1-based rank of the best-ranked gold candidate (0 if the query has no gold in the pool)"""
    S = mask_self(S, qids, cids)
    cidx = {c: i for i, c in enumerate(cids)}
    out = []
    for i, q in enumerate(qids):
        g = [cidx[c] for c in gold[q] if c in cidx and c != q]
        out.append(int((S[i] > S[i, g].max()).sum()) + 1 if g else 0)
    return out

def tfidf_scores(q_texts, c_texts):
    """classic TF-IDF cosine similarity, fitted on the pool only. [n_queries, n_candidates] float32"""
    from sklearn.feature_extraction.text import TfidfVectorizer

    def make(min_df, max_df):
        return TfidfVectorizer(sublinear_tf=True, ngram_range=C.TFIDF_NGRAM, min_df=min_df, max_df=max_df,
                               max_features=300_000, dtype=np.float32)

    try:
        vec = make(2, 0.9)
        X = vec.fit_transform(c_texts)
    except ValueError:                                              # tiny / degenerate pool (smoke runs)
        vec = make(1, 1.0)
        X = vec.fit_transform(c_texts)
    Q = vec.transform(q_texts)
    return (Q @ X.T).toarray().astype(np.float32)

def zscore_rows(S):
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-8)

def hybrid_scores(S_dense, S_tfidf, lam):
    return zscore_rows(S_dense) + lam * zscore_rows(S_tfidf)

def mine_hard_negs(S_tfidf, qids, cids, gold, pool_n):
    """for every query: the top `pool_n` TF-IDF candidates that are neither the query itself nor gold"""
    S = mask_self(S_tfidf, qids, cids)
    cidx = {c: i for i, c in enumerate(cids)}
    out = {}
    for i, q in enumerate(qids):
        for g in gold[q]:
            j = cidx.get(g)
            if j is not None:
                S[i, j] = -np.inf
        n = min(pool_n, S.shape[1] - 1)
        idx = np.argpartition(-S[i], n - 1)[:n]
        idx = idx[np.argsort(-S[i, idx])]
        out[q] = [cids[j] for j in idx if np.isfinite(S[i, j])]
    return out

def _selftest_mining():
    cids = ["x", "g1", "n1", "n2", "n3"]
    S = np.array([[0.99, 0.95, 0.9, 0.5, 0.4]], dtype=np.float32)    # "x" is itself, "g1" is gold
    hn = mine_hard_negs(S, ["x"], cids, {"x": {"g1"}}, 2)
    assert hn["x"] == ["n1", "n2"], hn
    return True

_selftest_mining()

def per_query_table(S, sp, preds, k):
    ranks = first_hit_rank(S, sp.qids, sp.cids, sp.gold)
    rows = []
    for i, q in enumerate(sp.qids):
        g = sp.gold[q]
        hits = len(set(preds[q][:k]) & g)
        rows.append({"qid": q, "n_gold": len(g), "hits": hits,
                     "precision": hits / k, "recall": hits / len(g) if g else 0.0,
                     "first_hit_rank": ranks[i], "q_tokens": len(sp.qdocs[i].ids),
                     "q_chunks": len(sp.qdocs[i].spans)})
    return pd.DataFrame(rows)

### 6. Adaptation method, retrieval encoder, document pooling and ranking loss

In [ ]:


# (maxsim: the chunk vectors are kept and two documents are scored by late interaction instead)

class LoRMALinear(nn.Module):
    """LoRMA - Low-Rank MULTIPLICATIVE Adaptation (Bihany et al., 2025).  LoRA adds a low-rank term to the frozen weight,
    LoRMA multiplies it:   W' = M W0   with the d x d matrix M = I + s*B*A  ("plus", B = 0 at init  ->  M = I)
                                                          or   M = Pi(B*A)   ("pi",   rank-inflating cyclic permutation).
    Implemented on the output of the frozen projection:  y = W0 x ;  y' = M y.  For "plus", M is never materialised."""

    def __init__(self, base, r, alpha, variant):
        super().__init__()
        self.base, self.variant = base, variant                     # base = the frozen W0
        dev = base.weight.device
        d = base.out_features                                       # M acts on the OUTPUT space of W0
        self.d, self.r = d, r
        if variant == "plus":
            self.scale = alpha / r
            self.A = nn.Parameter(torch.empty(r, d, device=dev))
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            self.B = nn.Parameter(torch.zeros(d, r, device=dev))
        elif variant == "pi":

            B = torch.randn(d, r) * 0.02
            B[:, 0] = 1.0
            A = torch.zeros(r, d)
            A[0, 0] = 1.0
            self.B, self.A = nn.Parameter(B.to(dev)), nn.Parameter(A.to(dev))
            ar = torch.arange(d, device=dev)
            self.register_buffer("rot_idx", (ar[None, :] - ar[:, None]) % d, persistent=False)   # rot[i, j] = M[i, (j - i) % d]
        else:
            raise ValueError(variant)

    def forward(self, x):
        y = self.base(x)                                            # W0 x
        if self.variant == "plus":
            return y + F.linear(F.linear(y, self.A), self.B) * self.scale
        with torch.autocast(device_type="cuda", enabled=False):
            rot = torch.gather(self.B.float() @ self.A.float(), 1, self.rot_idx)     # built in fp32: exact identity at init
        return F.linear(y, rot.to(y.dtype))

PEFT_NAME, PEFT_VARIANT = "LoRMA-plus", "plus"

def install_peft(enc):
    assert C.PEFT == "lorma_plus", "this notebook only contains the 'lorma_plus' implementation"
    names, n = set(C.LORA_TARGETS), 0
    for blk in enc.lm.enc_layers:
        for parent in (blk.attn, blk.ffn):
            for name, child in list(parent.named_children()):
                if name in names and isinstance(child, nn.Linear):
                    setattr(parent, name, LoRMALinear(child, C.LORA_R, C.LORA_ALPHA, PEFT_VARIANT))
                    n += 1
    assert n > 0, "no projection matched LORA_TARGETS"
    return n

class RetrievalEncoder(nn.Module):
    def __init__(self, lm, proj_dim=None):
        super().__init__()
        self.lm = lm
        d = lm.cfg.d_model
        p = proj_dim or d
        self.head = nn.Linear(d, p, bias=False)                     # identity at init -> zero-shot == plain pooling
        nn.init.eye_(self.head.weight)
        self.attn = nn.Linear(p, 1)                                 # chunk attention pooler (zero init = uniform mean)
        nn.init.zeros_(self.attn.weight)
        nn.init.zeros_(self.attn.bias)
        self.use_ckpt = False

    def hidden(self, ids, lens):
        """final-norm ENCODER states [B, T, d] of the (adapted) Model-1 encoder.  The encoder is BIDIRECTIONAL, so padding must be
        masked (right-padded batches then give the same states as encoding every window alone - checked in CELL 7)."""
        lm = self.lm
        keep = torch.arange(ids.size(1), device=ids.device)[None, :] < lens[:, None]
        mask = lm._mask4(keep)                                      # (B, 1, 1, T) bool, or None when nothing is padded
        x = lm.drop(lm.embed(ids))
        for layer in lm.enc_layers:
            if self.use_ckpt and self.training and torch.is_grad_enabled():
                x = grad_checkpoint(layer, x, lm.rope_cos, lm.rope_sin, mask, use_reentrant=False)
            else:
                x = layer(x, lm.rope_cos, lm.rope_sin, mask)
        return lm.enc_norm(x)

    def token_features(self, ids, lens, need):
        h = self.hidden(ids, lens).float()
        out = {}
        if "mean" in need:
            m = (torch.arange(ids.size(1), device=ids.device)[None, :] < lens[:, None]).unsqueeze(-1).float()
            out["mean"] = (h * m).sum(1) / lens[:, None].float()
        if "last" in need:
            out["last"] = h[torch.arange(ids.size(0), device=ids.device), lens - 1]
        return out

def chunk_features(enc, chunks, need, budget, train=False, use_amp=True):
    """
    chunks: list of 1-D int arrays (each <= CHUNK_LEN).  Returns {pool: [N, d] float32} in the ORIGINAL chunk order.
    Chunks are sorted by length so micro-batches carry little padding; each micro-batch holds <= `budget` tokens.
    train=True keeps the autograd graph (features stay on the GPU), otherwise features are moved to the CPU.
    """
    n = len(chunks)
    lens = np.array([len(c) for c in chunks], dtype=np.int64)
    order = np.argsort(-lens, kind="stable")
    outs = {k: [] for k in need}
    i = 0
    while i < n:
        L = int(lens[order[i]])
        mb = max(1, budget // max(L, 1))
        sel = order[i:i + mb]
        ids = np.zeros((len(sel), L), dtype=np.int64)
        for r, j in enumerate(sel):
            ids[r, :lens[j]] = chunks[j]
        ids_t = torch.from_numpy(ids).to(DEVICE)
        lens_t = torch.from_numpy(lens[sel]).to(DEVICE)
        with (amp_ctx() if use_amp else torch.autocast(device_type="cuda", enabled=False)):
            feats = enc.token_features(ids_t, lens_t, need)
        for k in need:
            outs[k].append(feats[k] if train else feats[k].float().cpu())
        i += mb
    inv = np.empty(n, dtype=np.int64)
    inv[order] = np.arange(n)
    res = {}
    for k in need:
        cat = torch.cat(outs[k], dim=0)
        res[k] = cat[torch.from_numpy(inv).to(cat.device)]
    return res

def docs_to_chunks(docs, train_chunks=None, rng=None):
    """flatten documents into chunks.  seg[i] = index of the document that chunk i belongs to (non-decreasing)."""
    chunks, seg = [], []
    for di, d in enumerate(docs):
        spans = d.spans
        if train_chunks and len(spans) > train_chunks:              # training: a random subset of the chunks
            pick = np.sort(rng.choice(len(spans), train_chunks, replace=False))
            spans = [spans[p] for p in pick]
        for s, e in spans:
            chunks.append(d.ids[s:e])
            seg.append(di)
    return chunks, np.asarray(seg, dtype=np.int64)

def assemble(enc, feat, seg, n_docs, doc_pool):
    """chunk features [N, d] -> head -> L2 -> pool the chunks of every document -> L2.   Differentiable."""
    z = F.normalize(enc.head(feat.float()), dim=-1)
    if doc_pool == "maxsim":                                        # late interaction: keep every chunk vector -> (P [n_docs, M, p], mask)
        return pad_chunks(z, seg, n_docs)
    p = z.size(1)
    if doc_pool == "first":
        counts = torch.bincount(seg, minlength=n_docs)
        starts = torch.cumsum(counts, 0) - counts
        out = z[starts]
    elif doc_pool == "max":
        idx = seg[:, None].expand(-1, p)
        out = z.new_full((n_docs, p), float("-inf")).scatter_reduce(0, idx, z, reduce="amax", include_self=False)
    elif doc_pool == "attn":
        s = enc.attn(z).squeeze(-1)
        w = torch.exp(s - s.max().detach())
        out = z.new_zeros(n_docs, p).index_add(0, seg, z * w[:, None])   # the per-doc denominator cancels in the L2 norm
    else:                                                           # "mean" (sum of unit vectors; L2 norm makes it a mean)
        out = z.new_zeros(n_docs, p).index_add(0, seg, z)
    return F.normalize(out, dim=-1)

def _selftest_assemble():
    """vectorised document pooling == a naive per-document loop"""
    g = torch.Generator().manual_seed(0)
    d = 8
    enc = types.SimpleNamespace(head=nn.Identity(), attn=nn.Linear(d, 1))
    nn.init.normal_(enc.attn.weight, std=0.5)
    feat = torch.randn(7, d, generator=g)
    seg = torch.tensor([0, 0, 0, 1, 2, 2, 3])                       # 4 docs with 3 / 1 / 2 / 1 chunks
    z = F.normalize(feat, dim=-1)
    for pool in ("mean", "max", "first", "attn"):
        got = assemble(enc, feat, seg, 4, pool)
        for di in range(4):
            zz = z[seg == di]
            if pool == "mean":
                ref = zz.mean(0)
            elif pool == "max":
                ref = zz.max(0).values
            elif pool == "first":
                ref = zz[0]
            else:
                w = torch.softmax(enc.attn(zz).squeeze(-1), 0)
                ref = (w[:, None] * zz).sum(0)
            ref = F.normalize(ref, dim=-1)
            assert torch.allclose(got[di], ref, atol=1e-5), (pool, di)
    return True

_selftest_assemble()

import contextlib, itertools

NEG = -1e4

def pad_chunks(z, seg, n_docs):
    """chunk vectors z [N, p] (grouped by document, seg non-decreasing) -> padded P [n_docs, M, p] + bool mask [n_docs, M]"""
    counts = torch.bincount(seg, minlength=n_docs)
    starts = torch.cumsum(counts, 0) - counts
    pos = torch.arange(z.size(0), device=z.device) - starts[seg]
    M = int(counts.max())
    P = z.new_zeros(n_docs, M, z.size(1)).index_put((seg, pos), z)           # out-of-place -> differentiable w.r.t. z
    mask = torch.arange(M, device=z.device)[None, :] < counts[:, None]
    return P, mask

def maxsim_block(Pq, Mq, Pc, Mc):
    """symmetric chunk-level MaxSim.  Pq [a, Mq, p] (mask Mq) vs Pc [b, Mc, p] (mask Mc) -> [a, b]:
       0.5 * ( mean over the query's chunks of the best candidate chunk + mean over the candidate's chunks of the best query chunk )"""
    sims = torch.einsum("amp,bkp->abmk", Pq, Pc)                             # [a, b, Mq, Mc]
    best_c = sims.masked_fill(~Mc[None, :, None, :], NEG).amax(-1)           # [a, b, Mq]
    mq = Mq[:, None, :].to(sims.dtype)
    s1 = (best_c * mq).sum(-1) / mq.sum(-1).clamp(min=1)
    best_q = sims.masked_fill(~Mq[:, None, :, None], NEG).amax(2)            # [a, b, Mc]
    mc = Mc[None, :, :].to(sims.dtype)
    s2 = (best_q * mc).sum(-1) / mc.sum(-1).clamp(min=1)
    return 0.5 * (s1 + s2)

def maxsim_all(Qrep, Crep, qb=None, numpy_out=False):
    """MaxSim of every query against every candidate; processed in query blocks so that the [a, b, Mq, Mc] tensor stays small"""
    (Pq, Mq), (Pc, Mc) = Qrep, Crep
    Pc, Mc = Pc.to(DEVICE), Mc.to(DEVICE)
    qb = qb or max(1, int(3e7 // max(1, Pc.size(0) * Pq.size(1) * Pc.size(1))))
    with (torch.no_grad() if numpy_out else contextlib.nullcontext()):
        S = torch.cat([maxsim_block(Pq[i:i + qb].to(DEVICE), Mq[i:i + qb].to(DEVICE), Pc, Mc)
                       for i in range(0, Pq.size(0), qb)], 0)
        return S.float().cpu().numpy() if numpy_out else S

def sim_t(Qrep, Crep):
    """differentiable similarity matrix [nq, nc]: cosine for single-vector documents, MaxSim for chunk-level ones"""
    return maxsim_all(Qrep, Crep) if isinstance(Qrep, tuple) else Qrep @ Crep.T

def sim_np(Qe, Ce):
    """numpy similarity matrix (inference)"""
    return maxsim_all(Qe, Ce, numpy_out=True) if isinstance(Qe, tuple) else Qe @ Ce.T

def backbone_groups(enc):
    """optimiser groups for the trainable backbone parameters: layer-wise LR decay (LAYER_DECAY ** distance-from-the-top),
    weight decay only on matrices, a separate LR multiplier for the RMSNorm gains"""
    L, buckets = len(enc.lm.enc_layers), {}
    for n, p in enc.named_parameters():
        if not (p.requires_grad and n.startswith("lm.")):
            continue
        m = re.match(r"lm\.enc_layers\.(\d+)\.", n)
        depth = int(m.group(1)) if m else (-1 if n.startswith("lm.embed") else L - 1)     # embeddings lowest, final norm top
        is_norm = ("norm" in n)
        buckets.setdefault((depth, p.ndim >= 2, is_norm), []).append(p)
    groups = []
    for (depth, mat, is_norm), ps in sorted(buckets.items()):
        lr = C.LR * (C.LAYER_DECAY ** (L - 1 - depth)) * (C.NORM_LR_MULT if is_norm else 1.0)
        groups.append({"params": ps, "lr": lr, "base_lr": lr, "weight_decay": C.BB_WD if mat else 0.0})
    return groups

def infonce_mp(S, pos, selfm):
    """multi-positive InfoNCE: every gold case competes ONLY against the non-gold candidates
       (in-batch + TF-IDF hard negatives); other gold cases and the query itself are masked out of the denominator."""
    logits = (S / C.TEMP).float().masked_fill(selfm, NEG)
    neg = ~pos & ~selfm
    lse_neg = torch.logsumexp(logits.masked_fill(~neg, NEG), dim=1, keepdim=True)
    per = F.softplus(lse_neg - logits) * pos                         # -log( e^{s_p} / (e^{s_p} + sum_n e^{s_n}) )
    q_loss = per.sum(1) / pos.sum(1).clamp(min=1)
    return q_loss[pos.any(1)].mean()

def margin_loss(S, pos, selfm):
    """pairwise hinge  max(0, m - s(q,pos) + s(q,neg))  over all (gold, non-gold) pairs of the batch:
       half batch-all (mean over violated pairs), half batch-hard (hardest negative of every gold case)"""
    m = C.MARGIN
    neg = ~pos & ~selfm
    valid = pos[:, :, None] & neg[:, None, :]                        # (query, gold, negative)
    viol = F.relu(m - S[:, :, None] + S[:, None, :]) * valid
    batch_all = viol.sum() / (viol > 0).sum().clamp(min=1)
    hardest = viol.max(dim=2).values                                 # hardest negative per gold case
    batch_hard = (hardest * pos).sum() / pos.sum().clamp(min=1)
    return 0.5 * batch_all + 0.5 * batch_hard

def rank_loss(S, pos, selfm):
    return (infonce_mp(S, pos, selfm) + C.COMBO_W * margin_loss(S, pos, selfm))

def _selftest_maxsim():
    """vectorised MaxSim == a naive loop over (query, candidate) pairs"""
    g = torch.Generator().manual_seed(1)
    seg_q, seg_c = torch.tensor([0, 0, 0, 1]), torch.tensor([0, 1, 1, 2, 2, 2])
    zq = F.normalize(torch.randn(4, 6, generator=g), dim=-1)
    zc = F.normalize(torch.randn(6, 6, generator=g), dim=-1)
    got = maxsim_all(pad_chunks(zq, seg_q, 2), pad_chunks(zc, seg_c, 3), qb=1).cpu()
    for a in range(2):
        for b in range(3):
            s = zq[seg_q == a] @ zc[seg_c == b].T
            ref = 0.5 * (s.max(1).values.mean() + s.max(0).values.mean())
            assert abs(float(got[a, b]) - float(ref)) < 1e-5, ("maxsim", a, b, float(got[a, b]), float(ref))
    zq.requires_grad_(True)                                                  # the padded gather must stay differentiable
    maxsim_all(pad_chunks(zq, seg_q, 2), pad_chunks(zc, seg_c, 3)).sum().backward()
    assert zq.grad is not None and float(zq.grad.abs().sum()) > 0
    return True

def _selftest_loss():
    """the ranking loss is finite, differentiable, lower for a correct ranking than for a reversed one and ignores the query itself"""
    pos = torch.tensor([[1, 0, 0, 0, 0], [0, 1, 1, 0, 0]], dtype=torch.bool)
    selfm = torch.tensor([[0, 0, 0, 0, 1], [0, 0, 0, 0, 1]], dtype=torch.bool)
    good = torch.tensor([[0.9, 0.1, 0.0, 0.2, 0.99], [0.1, 0.8, 0.7, 0.0, 0.99]])
    bad = torch.tensor([[0.1, 0.9, 0.8, 0.2, 0.99], [0.8, 0.1, 0.2, 0.7, 0.99]], requires_grad=True)
    lg, lb = rank_loss(good, pos, selfm), rank_loss(bad, pos, selfm)
    assert torch.isfinite(lg) and torch.isfinite(lb) and float(lg) < float(lb), (float(lg), float(lb))
    lb.backward()
    assert torch.isfinite(bad.grad).all() and float(bad.grad.abs().sum()) > 0
    assert float(bad.grad[:, 4].abs().sum()) == 0.0, "the query itself must not influence the loss"
    return True

_selftest_maxsim()
_selftest_loss()

log("self-tests passed: chunk spans, metrics, hard-negative mining, document pooling, MaxSim, ranking loss")

def build_encoder():
    lm = P1_MODEL.to(DEVICE)
    for p in lm.parameters():
        p.requires_grad_(False)
    enc = RetrievalEncoder(lm, C.PROJ_DIM).to(DEVICE)
    enc.use_ckpt = bool(C.GRAD_CKPT)
    return enc

def trainable_state(enc):
    return {n: p.detach().cpu().clone() for n, p in enc.named_parameters() if p.requires_grad}

def load_trainable_state(enc, state):
    missing, unexpected = enc.load_state_dict(state, strict=False)
    assert not unexpected, f"unexpected keys in the LoRA checkpoint: {unexpected[:5]}"
    own = {n for n, p in enc.named_parameters() if p.requires_grad}
    assert own <= set(state), f"checkpoint lacks trainable params: {sorted(own - set(state))[:5]}"

@torch.inference_mode()
def probe_vectors(enc, docs, n=4, max_len=256):
    """first-chunk mean features of a few documents, in full fp32 (used for the LoRA no-op check)"""
    was = enc.training
    enc.eval()
    ch = [d.ids[:max_len] for d in docs[:n]]
    f = chunk_features(enc, ch, ("mean",), 8192, train=False, use_amp=False)["mean"]
    enc.train(was)
    return f

@torch.inference_mode()
def encode_split_raw(enc, sp, need=("mean", "last")):
    """one pass of the LM over queries + pool of a split (all chunks).  Keeps the raw token-pooled chunk features."""
    was = enc.training
    enc.eval()
    docs = sp.qdocs + sp.cdocs
    chunks, seg = docs_to_chunks(docs)
    t0 = time.time()
    feats = chunk_features(enc, chunks, need, C.TOK_BUDGET_EVAL, train=False)
    enc.train(was)
    return {"feats": feats, "seg": seg, "n_docs": len(docs), "nq": len(sp.qdocs),
            "seconds": time.time() - t0, "n_chunks": len(chunks)}

@torch.inference_mode()
def embed_from_raw(enc, raw, token_pool, doc_pool, center="none"):
    """-> (query embeddings, candidate embeddings) as float32 numpy arrays, L2-normalised"""
    f = raw["feats"][token_pool].to(DEVICE)
    seg = torch.from_numpy(raw["seg"]).to(DEVICE)
    out = assemble(enc, f, seg, raw["n_docs"], doc_pool)
    if isinstance(out, tuple):                                      # "maxsim": padded chunk vectors (P, mask) per query / candidate
        P, M = out[0].float().cpu(), out[1].cpu()
        return (P[:raw["nq"]], M[:raw["nq"]]), (P[raw["nq"]:], M[raw["nq"]:])
    E = out.float().cpu().numpy()
    Qe, Ce = E[:raw["nq"]], E[raw["nq"]:]
    if center == "pool":                                            # unsupervised: statistics of the evaluated pool only
        mu = Ce.mean(0, keepdims=True)
        Qe, Ce = l2n(Qe - mu), l2n(Ce - mu)
    return Qe, Ce

def evaluate_dense(Qe, Ce, sp, k=None):
    k = k or C.K
    S = sim_np(Qe, Ce)
    preds = rank_topk(S, sp.qids, sp.cids, k)
    return micro_prf(preds, sp.gold, sp.qids, k), preds, S

### 7. Baselines, zero-shot Phase 1, adaptation install check, hard negatives

In [ ]:


ENC = build_encoder()
N_BASE_PARAMS = count_params(ENC.lm)
dev = SP["dev"]

def dev_grid(enc, raw, sp, token_pools=("mean", "last"), doc_pools=("mean", "max", "first", "maxsim"), centers=("none", "pool")):
    rows = []
    for tp in token_pools:
        for dp in doc_pools:
            for ce in centers:
                if dp == "maxsim" and ce != "none":
                    continue                                     # centring is only defined for single-vector documents
                Qe, Ce = embed_from_raw(enc, raw, tp, dp, ce)
                m, _, _ = evaluate_dense(Qe, Ce, sp)
                rows.append({"token_pool": tp, "doc_pool": dp, "center": ce,
                             "P@7": round(m["P"], 4), "R@7": round(m["R"], 4), "F1@7": round(m["F1"], 4)})
    return pd.DataFrame(rows).sort_values("F1@7", ascending=False).reset_index(drop=True)

t0 = time.time()
TFIDF_DEV = tfidf_scores([dev.qtext[q] for q in dev.qids], [dev.ctext[c] for c in dev.cids])
m_tfidf = micro_prf(rank_topk(TFIDF_DEV, dev.qids, dev.cids, C.K), dev.gold, dev.qids, C.K)
log(f"[baseline] TF-IDF dev  P@7={m_tfidf['P']:.4f}  R@7={m_tfidf['R']:.4f}  F1@7={m_tfidf['F1']:.4f}  "
    f"({fmt_time(time.time() - t0)})")
RESULTS["tfidf_dev"] = m_tfidf

PROBE_DOCS = dev.cdocs

@torch.inference_mode()
def padding_check(enc, docs):
    """right-padded, length-sorted micro-batches must give the same chunk features as encoding every chunk alone"""
    ch = [d.ids[:L] for d, L in zip(docs[:4], (200, 37, 120, 64))]
    enc_was = enc.training
    enc.eval()
    batch = chunk_features(enc, ch, ("mean", "last"), 8192, train=False, use_amp=False)
    worst = 0.0
    for i, c in enumerate(ch):
        one = chunk_features(enc, [c], ("mean", "last"), 8192, train=False, use_amp=False)
        for k in ("mean", "last"):
            worst = max(worst, float((batch[k][i] - one[k][0]).abs().max()))
    enc.train(enc_was)
    assert worst < 1e-3, f"padding / ordering changes the chunk features (max abs diff {worst:.2e})"
    return worst

log(f"padding + ordering check passed (max abs diff {padding_check(ENC, PROBE_DOCS):.1e})")
V0 = probe_vectors(ENC, PROBE_DOCS)                                  # fp32 features BEFORE LoRA exists
if C.RUN_ZERO_SHOT:
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    raw0 = encode_split_raw(ENC, dev)
    log(f"[zero-shot] encoded dev: {raw0['n_docs']:,} docs / {raw0['n_chunks']:,} chunks in "
        f"{fmt_time(raw0['seconds'])} ({raw0['n_docs'] / max(raw0['seconds'], 1e-9):.1f} docs/s)")
    GRID0 = dev_grid(ENC, raw0, dev)
    log("[zero-shot] Phase-1 Model 2, dev, no adaptation (token pooling x document pooling x centering):")
    log(GRID0.to_string(index=False))
    RESULTS["zero_shot_grid"] = GRID0.to_dict("records")
    RESULTS["efficiency_zero_shot"] = {
        "dev_docs": raw0["n_docs"], "dev_chunks": raw0["n_chunks"], "seconds": raw0["seconds"],
        "docs_per_s": raw0["n_docs"] / max(raw0["seconds"], 1e-9),
        "peak_gpu_mem_gb": (torch.cuda.max_memory_allocated() / 2**30) if DEVICE.type == "cuda" else None}
    del raw0
    gc.collect()

def _selftest_trainable_grads(enc):
    """one tiny forward/backward: gradients must reach the method's own backbone parameters (catches a detached graph in seconds)"""
    was = enc.training
    enc.train()
    ch = [d.ids[:48] for d in PROBE_DOCS[:3]]
    f = chunk_features(enc, ch, ("mean",), 4096, train=True)["mean"]
    (enc.head(f.float()) ** 2).sum().backward()
    bb = [p for n_, p in enc.named_parameters() if p.requires_grad and n_.startswith("lm.")]
    nz = sum(1 for p in bb if p.grad is not None and float(p.grad.abs().sum()) > 0)
    enc.zero_grad(set_to_none=True)
    enc.train(was)
    assert nz > 0, "no gradient reaches the adaptation parameters - the graph is detached"
    return nz, len(bb)

for p_ in ENC.lm.parameters():
    p_.requires_grad_(False)
n_inj = install_peft(ENC)                                            # defined in CELL 6; switches on the method's own parameters
for p_ in ENC.head.parameters():
    p_.requires_grad_(True)
for p_ in ENC.attn.parameters():
    p_.requires_grad_(C.DOC_POOL == "attn")                          # only trained if the attention pooler is used
V1 = probe_vectors(ENC, PROBE_DOCS)
noop = float((V1 - V0).abs().max())
assert noop < 1e-5, f"{PEFT_NAME} changed the Phase-1 output at initialisation (max abs diff {noop:.3e}) - it must be a no-op"
n_bb = sum(p.numel() for n_, p in ENC.named_parameters() if p.requires_grad and n_.startswith("lm."))
n_head = sum(p.numel() for n_, p in ENC.named_parameters() if p.requires_grad and not n_.startswith("lm."))
n_train = count_params(ENC, trainable_only=True)
assert n_train == n_bb + n_head and n_bb > 0
n_nz, n_bbt = _selftest_trainable_grads(ENC)
log(f"{PEFT_NAME} installed ({n_inj} modules / blocks); no-op check passed (diff {noop:.1e}); gradient check passed ({n_nz}/{n_bbt} tensors)")
log(f"params: Phase-1 base {N_BASE_PARAMS / 1e6:.2f}M | trainable backbone {n_bb / 1e6:.3f}M | head/pooler {n_head / 1e6:.3f}M | "
    f"trainable {n_train / 1e6:.3f}M = {100 * n_train / count_params(ENC):.2f}% of the model")
RESULTS["params"] = {"base": N_BASE_PARAMS, "backbone_trainable": n_bb, "head_and_pooler": n_head, "trainable": n_train,
                     "total": count_params(ENC), "modules": n_inj}

# 4. TF-IDF hard negatives for the training queries
tr = SP["train"]
t0 = time.time()
TFIDF_TRAIN = tfidf_scores([tr.qtext[q] for q in tr.qids], [tr.ctext[c] for c in tr.cids])
m_tr = micro_prf(rank_topk(TFIDF_TRAIN, tr.qids, tr.cids, C.K), tr.gold, tr.qids, C.K)
HARD_NEGS_MAP = mine_hard_negs(TFIDF_TRAIN, tr.qids, tr.cids, tr.gold, C.HARD_POOL)
log(f"[train] TF-IDF F1@7={m_tr['F1']:.4f}; hard-negative pools mined for {len(HARD_NEGS_MAP)} queries "
    f"(median {int(np.median([len(v) for v in HARD_NEGS_MAP.values()]))} per query) in {fmt_time(time.time() - t0)}")
RESULTS["tfidf_train"] = m_tr
del TFIDF_TRAIN
gc.collect()

### 8. Training (resumable)

In [ ]:


# per step: B queries -> N_POS sampled gold cases each + HARD_NEGS TF-IDF hard negatives each
# candidates of the whole batch are shared negatives for every query (in-batch negatives)

# the query itself is masked out. loss = rank_loss(similarity matrix, positives, self mask)

LATEST, BEST = os.path.join(CKPT_DIR, "p7_latest.pt"), os.path.join(CKPT_DIR, "p7_best.pt")
LATEST_META, BEST_META = os.path.join(CKPT_DIR, "p7_latest_meta.json"), os.path.join(CKPT_DIR, "p7_best_meta.json")
LOG_CSV = os.path.join(CKPT_DIR, "p7_train_log.csv")

def _gscaler():
    enabled = AMP["enabled"] and AMP["dtype"] == torch.float16
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled, init_scale=2.0 ** 12)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled, init_scale=2.0 ** 12)

def lr_scale(step, total, warm):
    if step < warm:
        return (step + 1) / max(warm, 1)
    prog = (step - warm) / max(1, total - warm)
    return C.MIN_LR_RATIO + (1 - C.MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * prog))

def embed_train(docs, rng):
    chunks, seg = docs_to_chunks(docs, C.TRAIN_CHUNKS, rng)
    f = chunk_features(ENC, chunks, (C.TOKEN_POOL,), C.TOK_BUDGET_TRAIN, train=True)[C.TOKEN_POOL]
    return assemble(ENC, f, torch.from_numpy(seg).to(DEVICE), len(docs), C.DOC_POOL)

def evaluate_dev_now():
    raw = encode_split_raw(ENC, SP["dev"], need=(C.TOKEN_POOL,))
    Qe, Ce = embed_from_raw(ENC, raw, C.TOKEN_POOL, C.DOC_POOL, C.CENTER)
    m, _, _ = evaluate_dense(Qe, Ce, SP["dev"])
    return m, raw["seconds"]

def find_resume():
    best = None
    for d in walk_find("p7_latest_meta.json", [CKPT_DIR, INPUT_ROOT]):
        meta = read_json(os.path.join(d, "p7_latest_meta.json"))
        if not meta or meta.get("tag") != C.P7_TAG or not os.path.exists(os.path.join(d, "p7_latest.pt")):
            continue
        if best is None or meta["epoch"] > best[1]["epoch"]:
            best = (d, meta)
    return best

def save_best(state, f1):
    atomic_torch_save({"trainable": trainable_state(ENC), "tag": C.P7_TAG, "dev_f1": f1, "epoch": state["epoch"]}, BEST)
    atomic_json({"tag": C.P7_TAG, "dev_f1": f1, "epoch": state["epoch"]}, BEST_META)

def run_training():
    tr = SP["train"]
    train_qs = [q for q in tr.qids if tr.gold_in_pool[q]]
    B = C.BATCH_Q
    steps_per_epoch = math.ceil(len(train_qs) / B)
    total_steps = steps_per_epoch * C.EPOCHS
    warm = max(1, int(C.WARMUP_FRAC * total_steps))
    log(f"train queries with gold in the pool: {len(train_qs)}/{len(tr.qids)} | {steps_per_epoch} steps/epoch x "
        f"{C.EPOCHS} epochs = {total_steps} steps | warm-up {warm}")

    bb_groups = backbone_groups(ENC)                                  # layer-wise LR decay etc. (see CELL 6)
    bb_params = [p for g in bb_groups for p in g["params"]]
    head_params = [p for n, p in ENC.named_parameters() if p.requires_grad and not n.startswith("lm.")]
    opt = torch.optim.AdamW(
        bb_groups + [{"params": head_params, "lr": C.HEAD_LR, "base_lr": C.HEAD_LR, "weight_decay": C.WEIGHT_DECAY}],
        betas=(0.9, 0.999))
    scaler = _gscaler()
    all_trainable = bb_params + head_params
    log(f"optimiser: {len(bb_groups)} backbone groups (lr {min(g['lr'] for g in bb_groups):.1e} .. {max(g['lr'] for g in bb_groups):.1e}) "
        f"+ head/pooler group (lr {C.HEAD_LR:.1e}) | loss {C.LOSS} | {C.N_POS} gold + {C.HARD_NEGS} hard negatives per query")

    state = {"epoch": 0, "gs": 0, "best_f1": -1.0, "best_epoch": -1, "bad": 0, "history": []}
    prev = find_resume()
    if prev is not None:
        d, meta = prev
        pl = torch.load(os.path.join(d, "p7_latest.pt"), map_location="cpu", weights_only=False)
        load_trainable_state(ENC, pl["trainable"])
        opt.load_state_dict(pl["opt"])
        scaler.load_state_dict(pl["scaler"])
        state = pl["state"]
        if os.path.abspath(d) != os.path.abspath(CKPT_DIR) and os.path.exists(os.path.join(d, "p7_best.pt")):
            shutil.copy(os.path.join(d, "p7_best.pt"), BEST)
            shutil.copy(os.path.join(d, "p7_best_meta.json"), BEST_META)
        log(f"RESUMED from {d}: {state['epoch']} epochs done, best dev F1@7 {state['best_f1']:.4f} (epoch {state['best_epoch']})")

    if state["best_f1"] < 0:                                          # epoch 0 = Phase 1 + untouched adapters
        m0, _ = evaluate_dev_now()
        state.update(best_f1=m0["F1"], best_epoch=0)
        save_best({"epoch": 0}, m0["F1"])
        state["history"].append({"epoch": 0, "train_loss": None, "dev_P": m0["P"], "dev_R": m0["R"], "dev_F1": m0["F1"]})
        log(f"epoch 0 (zero-shot, LoRA at init)  dev P@7={m0['P']:.4f} R@7={m0['R']:.4f} F1@7={m0['F1']:.4f}")

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t_train0, last_epoch_s, good_steps = time.time(), 0.0, 0
    docs_seen, t_docs = 0, 0.0

    for epoch in range(state["epoch"], C.EPOCHS):
        if (time.time() - T_START) / 3600 + 1.3 * last_epoch_s / 3600 > C.MAX_RUNTIME_H:
            log("time budget would be exceeded by another epoch -> stopping (state is saved, re-run to continue)")
            break
        t_ep = time.time()
        ENC.train()
        rng = np.random.default_rng(C.SEED + 1000 * epoch)
        perm = rng.permutation(len(train_qs))
        run_loss, run_acc, run_n = 0.0, 0.0, 0
        for step in range(steps_per_epoch):
            t_step = time.time()
            qb = [train_qs[i] for i in perm[step * B:(step + 1) * B]]
            cand_ids, cand_docs, seen, tgt = [], [], {}, []

            def add(c):
                if c not in seen:
                    seen[c] = len(cand_ids)
                    cand_ids.append(c)
                    cand_docs.append(tr.cdocs[tr.c_index[c]])
                return seen[c]

            for q in qb:                                              # N_POS gold cases per query
                g = tr.gold_in_pool[q]
                pick = rng.choice(len(g), min(C.N_POS, len(g)), replace=False)
                tgt.append(add(g[int(pick[0])]))
                for t in pick[1:]:
                    add(g[int(t)])
            if C.HARD_NEGS:
                for q in qb:                                          # TF-IDF hard negatives
                    hn = HARD_NEGS_MAP[q]
                    if hn:
                        for t in rng.choice(len(hn), min(C.HARD_NEGS, len(hn)), replace=False):
                            add(hn[int(t)])
            pos_np = np.zeros((len(qb), len(cand_ids)), dtype=bool)   # gold cases of query i (whoever sampled them) = positives
            self_np = np.zeros((len(qb), len(cand_ids)), dtype=bool)  # the query itself is neither positive nor negative
            for i, q in enumerate(qb):
                for j, c in enumerate(cand_ids):
                    if c == q:
                        self_np[i, j] = True
                    elif c in tr.gold[q]:
                        pos_np[i, j] = True

            q_docs = [tr.qdocs[tr.q_index[q]] for q in qb]
            Qe = embed_train(q_docs, rng)
            Ce = embed_train(cand_docs, rng)
            S = sim_t(Qe, Ce)                                         # [B, n_candidates]  cosine or MaxSim
            pos_t, self_t = torch.from_numpy(pos_np).to(DEVICE), torch.from_numpy(self_np).to(DEVICE)
            loss = rank_loss(S, pos_t, self_t)
            top1 = S.detach().masked_fill(self_t, NEG).argmax(1)

            f = lr_scale(state["gs"], total_steps, warm)
            for gr in opt.param_groups:
                gr["lr"] = gr["base_lr"] * f
            if not torch.isfinite(loss):
                log(f"  non-finite loss at step {state['gs']}, batch skipped")
                opt.zero_grad(set_to_none=True)
                state["gs"] += 1
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            gnorm = float(torch.nn.utils.clip_grad_norm_(all_trainable, C.GRAD_CLIP))
            if math.isfinite(gnorm) and gnorm > 0:
                good_steps += 1
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            state["gs"] += 1
            if state["gs"] == 25 and good_steps == 0:
                raise RuntimeError("25 steps without a finite non-zero gradient - the LoRA graph is detached or fp16 overflows")

            run_loss += float(loss)
            run_acc += float(pos_t[torch.arange(len(qb), device=DEVICE), top1].float().mean())
            run_n += 1
            docs_seen += len(q_docs) + len(cand_docs)
            t_docs += time.time() - t_step
            if (step + 1) % 10 == 0 or step + 1 == steps_per_epoch:
                log(f"[ep {epoch + 1}/{C.EPOCHS}] step {step + 1}/{steps_per_epoch} | loss {run_loss / max(run_n, 1):.4f} | "
                    f"top-1 is gold {run_acc / max(run_n, 1):.3f} | gnorm {gnorm:.2f} | lr(max) {max(g['lr'] for g in opt.param_groups):.2e} | "
                    f"{docs_seen / max(t_docs, 1e-9):.1f} docs/s | elapsed {fmt_time(time.time() - T_START)}")

        last_epoch_s = time.time() - t_ep
        state["epoch"] = epoch + 1
        row = {"epoch": epoch + 1, "train_loss": run_loss / max(run_n, 1), "train_acc": run_acc / max(run_n, 1),
               "epoch_seconds": last_epoch_s}
        if (epoch + 1) % C.EVAL_EVERY == 0 or epoch + 1 == C.EPOCHS:
            m, eval_s = evaluate_dev_now()
            row.update(dev_P=m["P"], dev_R=m["R"], dev_F1=m["F1"], dev_eval_seconds=eval_s)
            improved = m["F1"] > state["best_f1"] + 1e-6
            if improved:
                state.update(best_f1=m["F1"], best_epoch=epoch + 1, bad=0)
                save_best(state, m["F1"])
            else:
                state["bad"] += 1
            log(f"== epoch {epoch + 1}: train loss {row['train_loss']:.4f} | dev P@7={m['P']:.4f} R@7={m['R']:.4f} "
                f"F1@7={m['F1']:.4f} {'(new best)' if improved else ''} | best {state['best_f1']:.4f} @ epoch {state['best_epoch']} | "
                f"{fmt_time(last_epoch_s)}")
        state["history"].append(row)
        atomic_torch_save({"trainable": trainable_state(ENC), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                           "state": state, "tag": C.P7_TAG}, LATEST)
        atomic_json({"tag": C.P7_TAG, "epoch": state["epoch"], "best_f1": state["best_f1"]}, LATEST_META)
        pd.DataFrame(state["history"]).to_csv(LOG_CSV, index=False)
        if state["bad"] >= C.PATIENCE:
            log(f"early stopping: no dev improvement for {C.PATIENCE} evaluations")
            break

    pd.DataFrame(state["history"]).to_csv(LOG_CSV, index=False)
    RESULTS["training"] = {"history": state["history"], "best_dev_f1": state["best_f1"], "best_epoch": state["best_epoch"],
                           "train_seconds": time.time() - t_train0, "docs_per_s": docs_seen / max(t_docs, 1e-9),
                           "peak_gpu_mem_gb": (torch.cuda.max_memory_allocated() / 2 ** 30) if DEVICE.type == "cuda" else None}
    return state

if C.TRAIN:
    TRAIN_STATE = run_training()
else:
    TRAIN_STATE = None
    log("TRAIN=False -> skipping training")

cands = [p for p in (BEST,) if os.path.exists(p)]
if not cands:
    for d in walk_find("p7_best_meta.json", [INPUT_ROOT]):
        meta = read_json(os.path.join(d, "p7_best_meta.json"))
        if meta and meta.get("tag") == C.P7_TAG and os.path.exists(os.path.join(d, "p7_best.pt")):
            cands.append(os.path.join(d, "p7_best.pt"))
if cands:
    bl = torch.load(cands[0], map_location="cpu", weights_only=False)
    load_trainable_state(ENC, bl["trainable"])
    log(f"loaded best adapters from {cands[0]} (epoch {bl.get('epoch')}, dev F1@7 {bl.get('dev_f1', float('nan')):.4f})")
else:
    log("no trained adapters found - continuing with the zero-shot model")

### 8b. Multi-view fusion, pseudo-relevance feedback and optional ensembling (inference only, dev-tuned)
One pass of the adapted LM over a split gives the raw chunk vectors. From them we build several **document views** (chunk-mean, chunk-max, MaxSim),
z-score each view per query, and combine them with small integer weights chosen on dev. A view-blend is only accepted if it beats the best single view by `FUSE_MIN_GAIN`
(dev has just 118 queries, so a tiny gain is noise). Pseudo-relevance feedback then adds the average candidate-to-candidate similarity to the query's current top-k.
Everything is computed from the model's own vectors - no lexical score is involved.

In [ ]:


FUSE_VIEWS = ("mean", "maxsim", "max")

def zrow(S):
    """per-query z-score, so that views with different scales can be added"""
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-6)

def fuse_views(raw):
    """-> ({view: score matrix}, candidate-candidate cosine of the chunk-mean vectors) from one set of raw chunk features"""
    out = {}
    for v in FUSE_VIEWS:
        Qv, Cv = embed_from_raw(ENC, raw, C.TOKEN_POOL, v, "none")
        out[v] = sim_np(Qv, Cv)
    _, Cm = embed_from_raw(ENC, raw, C.TOKEN_POOL, "mean", "none")
    return out, (Cm @ Cm.T).astype(np.float32)

def prf_term(S, sim_cc, sp, k):
    """mean similarity of every candidate to the query's current top-k (the query itself excluded)"""
    top = np.argsort(-mask_self(S, sp.qids, sp.cids), axis=1)[:, :k]
    return np.stack([sim_cc[t].mean(0) for t in top])

def load_extra_scores(split):
    """score matrices (scores_*.npz) of OTHER finished runs; used only if their query / candidate order matches this run"""
    mats = []
    for pth in C.EXTRA_SCORES:
        try:
            ids = read_json(os.path.join(os.path.dirname(pth), "score_ids.json"))
            ok = bool(ids) and all(ids[s]["qids"] == SP[s].qids and ids[s]["cids"] == SP[s].cids for s in ("dev", "test"))
            if not ok:
                log(f"  skipping {pth}: id order differs from this run")
                continue
            mats.append(np.load(pth)[split].astype(np.float32))
        except Exception as e:
            log(f"  could not load {pth}: {e!r}")
    return mats

def fuse_combine(views, sim_cc, extra, sp, cfg):
    names = list(views)
    w = np.array([cfg["weights"][n] for n in names], dtype=np.float32)
    S = sum(wi * zrow(views[n]) for wi, n in zip(w, names)) / float(w.sum())
    if extra and cfg["extra_w"] > 0:
        S = S + cfg["extra_w"] * np.mean([zrow(x) for x in extra], axis=0)
    if cfg["prf"]:
        k, beta = cfg["prf"]
        S = zrow(S) + beta * zrow(prf_term(S, sim_cc, sp, k))
    return S.astype(np.float32)

def fuse_tune_dev(raw_dev, sp):
    """pick view weights, ensemble weight and PRF setting on dev (coarse grids + a minimum-gain rule against over-fitting)"""
    views, sim_cc = fuse_views(raw_dev)
    names = list(views)
    f1 = lambda S: micro_prf(rank_topk(S, sp.qids, sp.cids, C.K), sp.gold, sp.qids, C.K)["F1"]
    z = {n: zrow(views[n]) for n in names}
    single = {n: f1(z[n]) for n in names}
    log("  single views, dev F1@7: " + " | ".join(f"{n} {single[n]:.4f}" for n in names))
    best_single = max(single, key=single.get)
    cfg = {"weights": {n: int(n == best_single) for n in names}, "extra_w": 0.0, "prf": None}
    cur = single[best_single]
    grid = []
    for w in itertools.product((0, 1, 2), repeat=len(names)):
        if sum(w) == 0:
            continue
        S = sum(wi * z[n] for wi, n in zip(w, names)) / float(sum(w))
        grid.append((f1(S), w))
    f_blend, w_blend = max(grid)
    if f_blend - cur >= C.FUSE_MIN_GAIN:
        cfg["weights"], cur = dict(zip(names, w_blend)), f_blend
    log(f"  view weights {cfg['weights']} -> dev F1@7 {cur:.4f}")
    extra = load_extra_scores("dev")
    if extra:
        for ew in (0.5, 1.0, 2.0):
            f = f1(fuse_combine(views, sim_cc, extra, sp, {**cfg, "extra_w": ew}))
            if f - cur >= C.FUSE_MIN_GAIN:
                cfg["extra_w"], cur = ew, f
        log(f"  ensemble with {len(extra)} other run(s): weight {cfg['extra_w']} -> dev F1@7 {cur:.4f}")
    best = (cur, None)
    for k, beta in itertools.product((3, 5, 10), (0.25, 0.5, 1.0)):
        f = f1(fuse_combine(views, sim_cc, extra, sp, {**cfg, "prf": (k, beta)}))
        if f > best[0] + C.FUSE_MIN_GAIN:
            best = (f, (k, beta))
    cfg["prf"] = best[1]
    log(f"  pseudo-relevance feedback {cfg['prf']} -> dev F1@7 {best[0]:.4f}")
    cfg["dev_f1"] = float(best[0])
    return cfg, fuse_combine(views, sim_cc, extra, sp, cfg)

def fuse_apply(raw, sp, cfg, split):
    views, sim_cc = fuse_views(raw)
    return fuse_combine(views, sim_cc, load_extra_scores(split), sp, cfg)

def _selftest_fusion():
    """fusion arithmetic on small synthetic matrices: a one-hot weight vector reproduces the (z-scored) single view"""
    r = np.random.default_rng(0)
    sp_ = types.SimpleNamespace(qids=[f"q{i}" for i in range(4)], cids=[f"c{j}" for j in range(9)])
    views = {n: r.normal(size=(4, 9)).astype(np.float32) for n in FUSE_VIEWS}
    sim_cc = r.normal(size=(9, 9)).astype(np.float32)
    cfg = {"weights": {n: int(n == FUSE_VIEWS[1]) for n in FUSE_VIEWS}, "extra_w": 0.0, "prf": None}
    S = fuse_combine(views, sim_cc, [], sp_, cfg)
    assert np.allclose(S, zrow(views[FUSE_VIEWS[1]]), atol=1e-5)
    S2 = fuse_combine(views, sim_cc, [], sp_, {**cfg, "prf": (3, 0.5)})
    assert S2.shape == S.shape and np.isfinite(S2).all()
    return True

_selftest_fusion()
log("self-tests passed: fusion")

### 9. Dev evaluation, pooling analysis, hybrid analysis, failure analysis

In [ ]:


dev = SP["dev"]
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
raw_dev = encode_split_raw(ENC, dev)
RESULTS["efficiency_adapted"] = {
    "dev_docs": raw_dev["n_docs"], "dev_chunks": raw_dev["n_chunks"], "seconds": raw_dev["seconds"],
    "docs_per_s": raw_dev["n_docs"] / max(raw_dev["seconds"], 1e-9),
    "peak_gpu_mem_gb": (torch.cuda.max_memory_allocated() / 2 ** 30) if DEVICE.type == "cuda" else None,
    "note": "LoRA kept un-merged; merging B@A into W would remove the adapter overhead"}
log(f"[adapted] encoded dev in {fmt_time(raw_dev['seconds'])} ({RESULTS['efficiency_adapted']['docs_per_s']:.1f} docs/s)")

doc_pools = ("mean", "max", "first", "maxsim") + (("attn",) if C.DOC_POOL == "attn" else ())
GRID1 = dev_grid(ENC, raw_dev, dev, doc_pools=doc_pools)
log("[adapted] dev, every inference-time pooling variant (the model was TRAINED with "
    f"token_pool={C.TOKEN_POOL}, doc_pool={C.DOC_POOL}):")
log(GRID1.to_string(index=False))
RESULTS["adapted_grid"] = GRID1.to_dict("records")

cmp_ = {}
for ce in (("none",) if C.DOC_POOL == "maxsim" else ("none", "pool")):
    Qe_, Ce_ = embed_from_raw(ENC, raw_dev, C.TOKEN_POOL, C.DOC_POOL, ce)
    cmp_[ce] = evaluate_dense(Qe_, Ce_, dev)[0]["F1"]
FINAL_CENTER = max(cmp_, key=cmp_.get)
log(f"centring on dev: {cmp_} -> using '{FINAL_CENTER}'")

Qe_dev, Ce_dev = embed_from_raw(ENC, raw_dev, C.TOKEN_POOL, C.DOC_POOL, FINAL_CENTER)
S_dense_dev = sim_np(Qe_dev, Ce_dev)
FUSION = None
if C.FUSE:
    log("[fusion] tuning view weights / ensemble / pseudo-relevance feedback on dev:")
    FUSION, S_dense_dev = fuse_tune_dev(raw_dev, dev)
    log(f"[fusion] chosen: {FUSION}")
    RESULTS["fusion"] = FUSION

hy_rows = []
for lam in (0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 2.0):
    mm = micro_prf(rank_topk(hybrid_scores(S_dense_dev, TFIDF_DEV, lam), dev.qids, dev.cids, C.K), dev.gold, dev.qids, C.K)
    hy_rows.append({"lambda_tfidf": lam, "P@7": round(mm["P"], 4), "R@7": round(mm["R"], 4), "F1@7": round(mm["F1"], 4)})
HY = pd.DataFrame(hy_rows)
log("[adapted] hybrid ranking  z(dense) + lambda * z(TF-IDF)  on dev:")
log(HY.to_string(index=False))
RESULTS["hybrid_dev"] = hy_rows
BEST_LAMBDA = float(HY.loc[HY["F1@7"].idxmax(), "lambda_tfidf"])
FINAL_LAMBDA = BEST_LAMBDA if C.USE_HYBRID else None

S_final_dev = S_dense_dev if FINAL_LAMBDA is None else hybrid_scores(S_dense_dev, TFIDF_DEV, FINAL_LAMBDA)
preds_dev = rank_topk(S_final_dev, dev.qids, dev.cids, C.K)
M_DEV = micro_prf(preds_dev, dev.gold, dev.qids, C.K)
log(f"\n>>> FINAL dev (submission configuration: token_pool={C.TOKEN_POOL}, doc_pool={C.DOC_POOL}, centre={FINAL_CENTER}, "
    f"hybrid={FINAL_LAMBDA}): P@7={M_DEV['P']:.4f}  R@7={M_DEV['R']:.4f}  F1@7={M_DEV['F1']:.4f}")
RESULTS["dev_final"] = {**M_DEV, "token_pool": C.TOKEN_POOL, "doc_pool": C.DOC_POOL, "center": FINAL_CENTER,
                        "hybrid_lambda": FINAL_LAMBDA}

# per-query failure analysis
PQ = per_query_table(S_final_dev, dev, preds_dev, C.K)
PQ["f1"] = 2 * PQ["hits"] / (C.K + PQ["n_gold"])
PQ.to_csv(os.path.join(OUT_DIR, "dev_per_query.csv"), index=False)
n_zero = int((PQ["hits"] == 0).sum())
_med = PQ.loc[PQ.first_hit_rank > 0, "first_hit_rank"].median()
log(f"queries with 0 hits in the top-7: {n_zero}/{len(PQ)} ({100 * n_zero / len(PQ):.1f}%) | "
    f"median rank of the first gold case: {int(_med) if _med == _med else 'n/a'}")
try:
    PQ["len_bucket"] = pd.qcut(PQ["q_tokens"], q=min(5, PQ["q_tokens"].nunique()), duplicates="drop")
except ValueError:
    PQ["len_bucket"] = "all"
PQ["gold_bucket"] = pd.cut(PQ["n_gold"], bins=[0, 1, 3, 6, 10**6], labels=["1", "2-3", "4-6", "7+"])
log("\nby query length (tokens):")
log(PQ.groupby("len_bucket", observed=True).agg(queries=("qid", "count"), recall=("recall", "mean"),
                                               precision=("precision", "mean"), zero_hit=("hits", lambda s: (s == 0).mean())).round(3).to_string())
log("\nby number of gold cases:")
log(PQ.groupby("gold_bucket", observed=True).agg(queries=("qid", "count"), recall=("recall", "mean"),
                                                precision=("precision", "mean"), zero_hit=("hits", lambda s: (s == 0).mean())).round(3).to_string())
np.savez_compressed(os.path.join(OUT_DIR, "dev_scores.npz"), S=S_dense_dev.astype(np.float32),
                    qids=np.array(dev.qids), cids=np.array(dev.cids))        # model-only dev score matrix (before any TF-IDF blend)
if not isinstance(Qe_dev, tuple):
    np.savez_compressed(os.path.join(OUT_DIR, "dev_embeddings.npz"), Q=Qe_dev, C=Ce_dev,
                        qids=np.array(dev.qids), cids=np.array(dev.cids))    # reusable in later experiments
del raw_dev
gc.collect()

### 10. Test predictions and submission files

In [ ]:

# CELL 10 - test predictions -> prediction.json (+ submission.zip)

PRED_PATH = os.path.join(OUT_DIR, "prediction.json")
ZIP_PATH = os.path.join(OUT_DIR, "submission.zip")

if C.RUN_PREDICT:
    te = SP["test"]
    raw_te = encode_split_raw(ENC, te, need=(C.TOKEN_POOL,))
    Qe_te, Ce_te = embed_from_raw(ENC, raw_te, C.TOKEN_POOL, C.DOC_POOL, FINAL_CENTER)
    S_te = sim_np(Qe_te, Ce_te)
    if FUSION is not None:
        S_te = fuse_apply(raw_te, te, FUSION, "test")                # same recipe as tuned on dev
    S_te_model = S_te.copy()                                          # model-only scores (before any TF-IDF blend)
    if FINAL_LAMBDA is not None:
        S_te = hybrid_scores(S_te, tfidf_scores([te.qtext[q] for q in te.qids], [te.ctext[c] for c in te.cids]), FINAL_LAMBDA)
    pred_te = rank_topk(S_te, te.qids, te.cids, C.K)

    pool_te = set(te.cids)
    for q in te.qids:
        p = pred_te[q]
        assert len(p) == C.K, (q, len(p))
        assert len(set(p)) == C.K, f"duplicate ids for query {q}"
        assert all(c in pool_te for c in p), f"id outside the test pool for query {q}"
        assert q not in p, f"query {q} retrieved itself"

    with open(PRED_PATH, "w", encoding="utf-8") as f:
        json.dump({q: pred_te[q] for q in te.qids}, f, indent=2)
    chk = json.load(open(PRED_PATH, encoding="utf-8"))
    assert list(chk.keys()) == te.qids and all(len(v) == C.K and len(set(v)) == C.K for v in chk.values())
    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(PRED_PATH, arcname="prediction.json")
    np.savez_compressed(os.path.join(OUT_DIR, "test_scores.npz"), S=S_te_model.astype(np.float32),
                        qids=np.array(te.qids), cids=np.array(te.cids))

    np.savez_compressed(os.path.join(OUT_DIR, f"scores_{C.EXP_NAME}.npz"), dev=S_dense_dev.astype(np.float32), test=S_te_model.astype(np.float32))
    atomic_json({s: {"qids": SP[s].qids, "cids": SP[s].cids} for s in ("dev", "test")}, os.path.join(OUT_DIR, "score_ids.json"))
    if not isinstance(Qe_te, tuple):
        np.savez_compressed(os.path.join(OUT_DIR, "test_embeddings.npz"), Q=Qe_te, C=Ce_te,
                            qids=np.array(te.qids), cids=np.array(te.cids))
    log(f"wrote {PRED_PATH}: {len(chk)} test queries x {C.K} ids | {ZIP_PATH}")
    if C.MAX_QUERIES or C.MAX_POOL:
        log("WARNING: this was a smoke run (MAX_QUERIES / MAX_POOL set) - the file covers only part of the test set. DO NOT SUBMIT IT.")
    log("example:", list(chk.items())[0])
    RESULTS["test"] = {"queries": len(chk), "pool": len(te.cids), "encode_seconds": raw_te["seconds"]}
    del raw_te
    gc.collect()
else:
    log("RUN_PREDICT=False -> no prediction file written")

### 11. Summary for the report

In [ ]:

# CELL 11 - summary table for the report + results.json

def _best_zero_shot():
    g = RESULTS.get("zero_shot_grid")
    if not g:
        return None
    return max(g, key=lambda r: r["F1@7"])

zs = _best_zero_shot()
rows = [{"system": "TF-IDF (classical baseline)", "P@7": RESULTS["tfidf_dev"]["P"], "R@7": RESULTS["tfidf_dev"]["R"],
         "F1@7": RESULTS["tfidf_dev"]["F1"], "trainable params": 0}]
if zs:
    rows.append({"system": f"Phase-1 zero-shot (best pooling: {zs['token_pool']}/{zs['doc_pool']}/centre={zs['center']})",
                 "P@7": zs["P@7"], "R@7": zs["R@7"], "F1@7": zs["F1@7"], "trainable params": 0})
rows.append({"system": "M1: Phase-1 MODEL 1 encoder + LoRMA+(r=32) | 512-tok windows stride 384 <=48, views mean+MaxSim+max fused, PRF | InfoNCE + margin",
             "P@7": M_DEV["P"], "R@7": M_DEV["R"], "F1@7": M_DEV["F1"], "trainable params": RESULTS["params"]["trainable"]})
SUMMARY = pd.DataFrame(rows)
log(SUMMARY.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

ea = RESULTS.get("efficiency_adapted", {})
log(f"\nmodel: {RESULTS['params']['base'] / 1e6:.1f}M Phase-1 params + {RESULTS['params']['trainable'] / 1e6:.2f}M trainable "
    f"({100 * RESULTS['params']['trainable'] / RESULTS['params']['total']:.2f}%) | "
    f"inference {ea.get('docs_per_s', float('nan')):.1f} docs/s | peak GPU memory (inference) {ea.get('peak_gpu_mem_gb')} GB | "
    f"training peak {RESULTS.get('training', {}).get('peak_gpu_mem_gb')} GB")
RESULTS["summary"] = rows
RESULTS["total_runtime_s"] = time.time() - T_START

def _jsonable(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (set, tuple)):
        return list(o)
    return str(o)

with open(os.path.join(OUT_DIR, f"results_{C.EXP_NAME}.json"), "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, indent=1, default=_jsonable)
for fn in ("p7_best.pt", "p7_train_log.csv"):                       # the adapters are tiny: keep them next to the outputs
    src = os.path.join(CKPT_DIR, fn)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(OUT_DIR, fn))
log(f"\nartifacts in {OUT_DIR}: " + ", ".join(sorted(os.listdir(OUT_DIR))))
log(f"total runtime {fmt_time(time.time() - T_START)}")